## 01-Classical Classifier Baseline
In this notebook, we will do a simple baseline classification using classicial method (Tfidf and logistic regression)

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [2]:
# load train and dev set

# please point to correct downloaded transcribed folders of the original dataset
data_path = "../../../data/" 

train_fname = data_path + "pitt/pitt_train_txt.csv"
dev_fname = data_path + "pitt/pitt_dev_txt.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 398
Dev df = 80


In [3]:
train_df.head()

,file_id,text,target,id,split_name
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,1,train
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,1,train
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,3,train
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,5,train
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,5,train


In [4]:
dev_df.head()

,file_id,text,target,id,split_name
0,122-0,"well, why <the mother is> &-um <she's not wit...",AD,122,dev
1,122-1,I see a tad bit .\nand somebody is doin(g) dr...,AD,122,dev
2,125-0,cookie jar .\ncookie jar .\n(..) plate .\n(..)...,AD,125,dev
3,134-0,well ‡ there's a boy trying to get some cookie...,AD,134,dev
4,134-1,a little boy is <tryin(g) to get the> &-uh ge...,AD,134,dev


### Pre-processing
We will lowercase the text for better efficiency

In [5]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

In [6]:
train_df.head()

,file_id,text,target,id,split_name,text_lower
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,1,train,mhm .\nalright .\nthere's &-um a young boy tha...
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,1,train,mhm .\nthere's a young boy &-uh going in a coo...
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,3,train,here's a cookie jar .\nand the lid is off the ...
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,5,train,the boy is slipping off the stool .\nhe's tryi...
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,5,train,okay ‡ he's fallin(g) off a chair .\nshe's &-...


In [7]:
dev_df.head()

,file_id,text,target,id,split_name,text_lower
0,122-0,"well, why <the mother is> &-um <she's not wit...",AD,122,dev,"well, why <the mother is> &-um <she's not wit..."
1,122-1,I see a tad bit .\nand somebody is doin(g) dr...,AD,122,dev,i see a tad bit .\nand somebody is doin(g) dr...
2,125-0,cookie jar .\ncookie jar .\n(..) plate .\n(..)...,AD,125,dev,cookie jar .\ncookie jar .\n(..) plate .\n(..)...
3,134-0,well ‡ there's a boy trying to get some cookie...,AD,134,dev,well ‡ there's a boy trying to get some cookie...
4,134-1,a little boy is <tryin(g) to get the> &-uh ge...,AD,134,dev,a little boy is <tryin(g) to get the> &-uh ge...


### Build Tfidf Vectorizer

We can use Tfidf Vectorizer to convert text to a fixed-length vector

In [8]:
vectorizer = TfidfVectorizer()


corpus = train_df["text_lower"]
print(len(corpus))
print("Building Tfidf Vectorizer...")
X_train = vectorizer.fit_transform(corpus)
print("Completed")

398
Building Tfidf Vectorizer...
Completed


In [9]:
print(X_train.shape)

(398, 1539)


### Train Logistic regression using Tfidf vectorizer

We can now train a simple logistic regression classifier using processed X_train

In [10]:
X_train = vectorizer.fit_transform(train_df["text_lower"])
y_train = train_df["target"]

print("Training logistic classifier...")
clf = LogisticRegression(random_state=42, 
                         # class_weight="balanced",
                        ).fit(X_train, y_train)
print("Completed")

Training logistic classifier...
Completed


In [11]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 86.93 %


### Get accuracy using cross validation

In [12]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=5)
print("Completed")
print(scores)
print("Average =", np.average(scores))

Performing cross-validation...
Completed
[0.725      0.7125     0.75       0.6835443  0.81012658]
Average = 0.7362341772151899


In [13]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.85      0.92      0.88       218
          CN       0.89      0.81      0.85       180

    accuracy                           0.87       398
   macro avg       0.87      0.86      0.87       398
weighted avg       0.87      0.87      0.87       398



### Now we can evaluate on dev set

In [14]:
X_dev = vectorizer.transform(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (80, 1539)
y_dev len = 80


In [15]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 83.75 %


In [16]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.85      0.87      0.86        46
          CN       0.82      0.79      0.81        34

    accuracy                           0.84        80
   macro avg       0.83      0.83      0.83        80
weighted avg       0.84      0.84      0.84        80



In [17]:
### Process prediction for test set

In [18]:
test_fname = data_path + "pitt/pitt_test_txt.csv"

test_df = pd.read_csv(test_fname)
print("Test df =", len(test_df))
test_df["text_lower"] = test_df["text"].apply(lambda text: text.lower().strip())
test_df.head()

Test df = 71


,file_id,text,target,id,split_name,text_lower
0,127-0,(..) well ‡ the mother's washing the dishes .\...,AD,127,test,(..) well ‡ the mother's washing the dishes .\...
1,181-0,well ‡ they they look like they're stealin(g)...,AD,181,test,well ‡ they they look like they're stealin(g)...
2,181-1,a little boy is gettin(g) himself some cookies...,AD,181,test,a little boy is gettin(g) himself some cookies...
3,181-2,uh she's &-um &+w &+w &+w washing dishes .\nan...,AD,181,test,uh she's &-um &+w &+w &+w washing dishes .\nan...
4,181-3,uh &+th these two little &+g kids are gettin(g...,AD,181,test,uh &+th these two little &+g kids are gettin(g...


In [19]:
X_test = vectorizer.transform(test_df["text_lower"])
test_df["prediction"] = clf.predict(X_test)
test_df.head()

,file_id,text,target,id,split_name,text_lower,prediction
0,127-0,(..) well ‡ the mother's washing the dishes .\...,AD,127,test,(..) well ‡ the mother's washing the dishes .\...,AD
1,181-0,well ‡ they they look like they're stealin(g)...,AD,181,test,well ‡ they they look like they're stealin(g)...,AD
2,181-1,a little boy is gettin(g) himself some cookies...,AD,181,test,a little boy is gettin(g) himself some cookies...,AD
3,181-2,uh she's &-um &+w &+w &+w washing dishes .\nan...,AD,181,test,uh she's &-um &+w &+w &+w washing dishes .\nan...,AD
4,181-3,uh &+th these two little &+g kids are gettin(g...,AD,181,test,uh &+th these two little &+g kids are gettin(g...,AD


In [20]:
correct_count = len(test_df.query("target == prediction"))
print(f"Correct count = {correct_count}, accuracy = {correct_count*100/len(test_df):.0f} %")

Correct count = 63, accuracy = 89 %


In [21]:
print(classification_report(test_df["target"], test_df["prediction"]))

              precision    recall  f1-score   support

          AD       0.85      0.98      0.91        42
          CN       0.96      0.76      0.85        29

    accuracy                           0.89        71
   macro avg       0.91      0.87      0.88        71
weighted avg       0.90      0.89      0.88        71

